# Makemore Part 1: Building a Bigram Character-Level Language Model

> Based on Andrej Karpathy's "Zero to Hero" series. This notebook lets you learn by running the code yourself.

**What you'll learn:**
- What a character-level language model is
- How to build a bigram model by counting
- How to reformulate it as a neural network
- How to train with gradient descent

---

## Table of Contents
1. [Introduction: What is Makemore?](#1-introduction)
2. [Loading and Exploring the Dataset](#2-dataset)
3. [Understanding Bigrams](#3-bigrams)
4. [Counting Bigrams with PyTorch Tensors](#4-counting)
5. [Visualizing the Bigram Matrix](#5-visualization)
6. [From Counts to Probabilities](#6-probabilities)
7. [Sampling Names from the Model](#7-sampling)
8. [Evaluating Model Quality: Loss Functions](#8-loss)
9. [Reformulating as a Neural Network](#9-neural-net)
10. [Training with Gradient Descent](#10-training)
11. [Sampling from the Trained Network](#11-neural-sampling)
12. [Conclusion & Next Steps](#12-conclusion)

<a id='1-introduction'></a>
## 1. Introduction: What is Makemore?

**Makemore**, as the name suggests, makes more of things that you give it. Train it on a dataset of names, and it will learn to make more things that sound name-like but are actually unique names. Maybe if you're looking for a cool, unique baby name, makemore might help you!

Here are some example generations from the neural network once we train it: *Dontel*, *Irot*, *Zhendi*... These all sound name-like but they're not actual names.

### What is a Character-Level Language Model?

Under the hood, makemore is a **character-level language model**. This means:
- It treats every single line (name) as an example
- Within each example, it treats them as **sequences of individual characters**
- For example, "reese" is the sequence: r, e, e, s, e

The model learns to **predict the next character in a sequence**. That's the fundamental task.

### What's Coming in This Series

We're going to implement a large number of character-level language models, starting simple and building up:

1. **Bigram models** (this notebook) - very simple, only looks at one previous character
2. **MLP models** - multilayer perceptrons with more context
3. **RNN models** - recurrent neural networks  
4. **Transformers** - the modern architecture behind GPT

By the end of the series, we'll build a transformer that is basically equivalent to **GPT-2**. That's kind of a big deal - it's a modern network, and you'll understand how it works on the level of characters.

### Two Approaches, Same Result

In this notebook, we'll build the same model two completely different ways:
1. **Counting-based**: Just count character pairs and normalize
2. **Neural network**: Use gradient descent to learn the same statistics

Both give the exact same result - but the neural network approach is much more flexible and will scale to the complex models coming later.

<a id='2-dataset'></a>
## 2. Loading and Exploring the Dataset

Our dataset is `names.txt` - a very large dataset of names that I found randomly on a government website. There are about 32,000 names in it.

We'll load it by reading the entire file as a string, then splitting it into individual words (one name per line).

In [ ]:
# Load the dataset
words = open('names.txt', 'r').read().splitlines()

In [ ]:
# Look at the first 10 names
words[:10]

In [ ]:
# How many names do we have?
len(words)

In [ ]:
# Shortest and longest names
print(f"Shortest name: {min(len(w) for w in words)} characters")
print(f"Longest name: {max(len(w) for w in words)} characters")

<a id='3-bigrams'></a>
## 3. Understanding Bigrams

Now let's think through our very first language model. A character-level language model predicts the next character in a sequence given some concrete sequence of characters before it.

### Every Word Contains Multiple Examples

Here's an important realization: every single word like "isabella" is actually **quite a few examples packed into that single word**. What is the existence of a word like "isabella" in the dataset telling us?

- The character 'i' is very likely to come **first** in a name
- The character 's' is likely to come after 'i'
- The character 'a' is likely to come after 'is'
- The character 'b' is very likely to come after 'isa'
- ...and so on, all the way to 'a' following 'isabell'

And there's **one more example** packed in here: after "isabella", the word is very likely to **end**. That's an explicit piece of information we have to be careful with.

So there's a lot packed into a single individual word in terms of statistical structure. And we don't have just one word - we have 32,000 of them!

### The Bigram Model

In a **bigram language model**, we're always working with just two characters at a time. We only look at **one character** that we're given, and we try to predict the next character in the sequence.

- What characters are likely to follow 'a'?
- What characters are likely to follow 'q'?

We're modeling this local structure but forgetting that we may have more information. We're always just looking at the previous character to predict the next one. It's a very simple and weak language model, but it's a great place to start.

### Special Start/End Tokens

We need to represent "this is the start of a name" and "this is the end of a name". We'll use a special `.` character for both. So "emma" becomes:
- `.emma.`
- Bigrams: (., e), (e, m), (m, m), (m, a), (a, .)

### The zip() Trick

A cute way to iterate over consecutive pairs in Python:

In [ ]:
# Let's look at the bigrams in "emma"
w = "emma"
chs = ['.'] + list(w) + ['.']
for ch1, ch2 in zip(chs, chs[1:]):
    print(f"({ch1}, {ch2})")

> **How the zip trick works**: If `chs = ['.', 'e', 'm', 'm', 'a', '.']`, then `chs[1:] = ['e', 'm', 'm', 'a', '.']`. The `zip()` function takes two iterators and pairs them up, creating tuples of consecutive entries. If one list is shorter, zip just halts when it runs out - that's why we get exactly the pairs we want.

<a id='4-counting'></a>
## 4. Counting Bigrams with PyTorch Tensors

To learn the statistics about which characters are likely to follow other characters, the simplest way in the bigram language model is to simply **count** how often any one of these combinations occurs in the training set.

### Why PyTorch Tensors?

It's going to be significantly more convenient for us to keep this information in a **2D array** instead of a Python dictionary:
- **Rows** = first character of the bigram
- **Columns** = second character of the bigram  
- **Entry at [i,j]** = how often character i is followed by character j

We'll use **PyTorch tensors** for this. PyTorch is a deep learning framework, but it also gives us efficient multi-dimensional arrays. We'll need this later when we build the neural network.

### Character-to-Integer Mapping

We need a lookup table from characters to integers (to index into our array). We'll create:
- `stoi` (string-to-integer): maps 'a' → 1, 'b' → 2, ..., 'z' → 26, '.' → 0
- `itos` (integer-to-string): the reverse mapping

I like having the special `.` token at position 0, with letters offset by 1.

In [ ]:
import torch

In [ ]:
# Create character-to-index and index-to-character mappings
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}  # a=1, b=2, ..., z=26
stoi['.'] = 0  # special start/end token = 0
itos = {i:s for s, i in stoi.items()}

print(f"Vocabulary size: {len(stoi)} characters")
print(f"stoi: {stoi}")

In [ ]:
# Create the count matrix N
# N[i, j] = count of character i followed by character j
N = torch.zeros((27, 27), dtype=torch.int32)

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

print(f"Total bigram count: {N.sum().item()}")

<a id='5-visualization'></a>
## 5. Visualizing the Bigram Matrix

Let's visualize this array. Each cell shows the bigram characters and how many times that bigram occurs in the dataset.

**Reading the visualization:**
- The **first row** (`.` row) shows counts for first letters - how often each letter starts a word
- The **first column** (`.` column) shows counts for last letters - how often each letter ends a word
- The **interior** shows which characters follow which in the middle of words
- **Darker blue** = more frequent

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16, 16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off')
plt.title('Bigram Counts: Row = first char, Column = second char');

> **What you can see:**
> - 'n' is very often an ending character (first column, row 'n')
> - 'n' almost always follows 'a' - that's a very likely combination
> - Some bigrams like 'qx' never occur (count = 0)

This array has all the information necessary to sample from our bigram language model!

<a id='6-probabilities'></a>
## 6. From Counts to Probabilities

These counts are telling us how often any one of these characters is to follow another. To sample, we need to convert them to **probability distributions** - we need each row to sum to 1.

### Model Smoothing

Before we normalize, we'll add **+1 to all counts**. Why?

Some bigrams (like "qj") never appear in training data, giving them probability = 0. If we ever encounter such a bigram later, we'd get infinite loss (log of 0 is negative infinity)!

Adding fake counts ensures no probability is exactly zero:
- The more you add, the more **uniform** the distribution becomes
- Adding +1 is gentle smoothing
- Adding +1,000,000 would make everything nearly uniform

### The Broadcasting Bug (Important!)

I want to **scare you a little bit** here. You really need to understand **broadcasting** and treat it with respect. It's not something to play fast and loose with.

When we normalize, we want to divide each row by its sum. Here's the trap:

In [ ]:
# Convert counts to probabilities with add-one smoothing
P = (N + 1).float()
P /= P.sum(1, keepdims=True)  # normalize each row

print(f"Shape of P: {P.shape}")
print(f"Each row sums to: {P[0].sum().item():.4f}")

> ### Why `keepdims=True` is Crucial
>
> **Without keepdims:**
> ```python
> P.sum(1)  # shape: (27,) - a 1D vector!
> ```
> When you divide a (27, 27) array by a (27,) vector, broadcasting rules say:
> 1. Align dimensions from the right
> 2. The (27,) becomes (1, 27) - a **row vector**
> 3. This row vector gets copied **vertically** 27 times
> 4. You end up normalizing **columns** instead of rows!
>
> **With keepdims=True:**
> ```python
> P.sum(1, keepdims=True)  # shape: (27, 1) - a column vector
> ```
> Now broadcasting copies horizontally, which correctly normalizes each row.
>
> **This is a subtle bug** - the code runs without error but gives completely wrong results. The fix is just one parameter, but missing it means your model is broken in a way that's very hard to debug.
>
> I strongly encourage you to read through the [PyTorch broadcasting semantics](https://pytorch.org/docs/stable/notes/broadcasting.html), practice it, and be careful with it.

In [ ]:
# Verify: probabilities for what follows 'a'
row_a = P[stoi['a']]
print(f"Probabilities for characters following 'a':")
print(f"Sum: {row_a.sum().item():.4f}")

<a id='7-sampling'></a>
## 7. Sampling Names from the Model

Now we can **generate names**! The algorithm:
1. Start with the `.` token (index 0) - the start token
2. Look up the probability row for the current character
3. Sample the next character using `torch.multinomial`
4. If we sample `.` again (index 0), that's the end token - we're done
5. Otherwise, continue from step 2 with the new character

### How `torch.multinomial` Works

`torch.multinomial` takes probabilities and returns integers sampled according to that distribution:
- Give it `[0.6, 0.3, 0.1]` and ask for 20 samples
- You'll get roughly 60% zeros, 30% ones, and 10% twos
- `replacement=True` means we can sample the same index multiple times

We use a `Generator` object with a fixed seed so results are reproducible - you'll get the exact same names I get.

In [ ]:
# Generate 10 names from our counting-based model
g = torch.Generator().manual_seed(2147483647)

print("Generated names:")
for i in range(10):
    out = []
    ix = 0  # start with '.'
    while True:
        p = P[ix]  # probability distribution for next char
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:  # sampled '.', end of name
            break
    print(''.join(out[:-1]))  # remove the trailing '.'

> **Why are these names so bad?**
>
> I'll be honest - when I first ran this, I spent a few minutes convincing myself it was actually working! The reason these samples are so terrible is that the **bigram language model is just really terrible**.
>
> Think about it from the model's perspective: when it generates "h" as a complete name, it doesn't know that 'h' is the *only* character. All it knows is:
> - 'h' was the previous character
> - How likely is 'h' to be followed by the end token?
> - Well, it's somewhat likely, so it ends the word
>
> The model has no memory of what came before. It doesn't know there were no other characters. That's why it generates nonsense like single-letter names.
>
> **Comparison to random**: If we used a uniform distribution (everything equally likely), we'd get complete garbage. The bigram model is at least *more* name-like - you can see it's doing something. It's just that bigrams alone aren't enough context.

<a id='8-loss'></a>
## 8. Evaluating Model Quality: Loss Functions

Now I'd like to somehow evaluate the quality of this model - summarize it into a **single number**. How good is it at predicting the training set?

### Likelihood and Log Likelihood

For each bigram in our dataset, the model assigns a probability. We can look at these probabilities:

- **Good model** → assigns high probability to actual bigrams
- **Bad model** → assigns low probability

As a measuring stick: with 27 possible characters, if everything was equally likely, you'd expect ~**4% probability** for each. Anything above 4% means we've learned something useful. Some of our probabilities are as high as 35-40%!

### Why Log Likelihood?

The **likelihood** is the product of all assigned probabilities. But this is unwieldy:
- All probabilities are between 0 and 1
- Multiplying many of them gives a *tiny* number

So we use **log likelihood** instead:
- `log(a × b × c) = log(a) + log(b) + log(c)`
- Products become sums - much more manageable!

The log function is **monotonic**: log(1) = 0, and as probability decreases, log becomes more negative (toward -∞ at probability 0).

### Negative Log Likelihood (NLL)

We want a **loss function** where lower is better. Since log likelihood is negative (and more negative = worse), we just flip the sign:

- **Negative log likelihood** = our loss
- **Lower NLL** = better model
- **Minimize NLL** = maximize likelihood

This is the foundation of statistical modeling.

In [ ]:
# Compute the negative log likelihood of our dataset under model P
log_likelihood = 0.0
n = 0

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1

nll = -log_likelihood
avg_nll = nll / n

print(f"Total negative log likelihood: {nll.item():.2f}")
print(f"Average NLL per bigram: {avg_nll.item():.4f}")

> **Interpreting the loss:**
> - Average NLL of ~2.45 means `e^2.45 ≈ 11.6`
> - The model is roughly as good as randomly guessing from ~12 characters
> - A random model (27 equally likely chars) would have NLL of `log(27) ≈ 3.3`
> - So we're doing better than random, but there's room to improve!
>
> **Note on smoothing:** Remember we added +1 to all counts? That's why some probabilities aren't exactly what the raw counts would give. This prevents any probability from being exactly 0 (which would give infinite loss).

<a id='9-neural-net'></a>
## 9. Reformulating as a Neural Network

We've arrived at our bigram model explicitly by doing something that felt sensible - counting and normalizing. Now I'd like to take an **alternative approach** that will look very different but end up in a very similar spot.

I want to cast the bigram problem into the **neural network framework**.

### The Neural Network Setup

Our neural network will:
- **Input**: A single character (as an integer)
- **Process**: A neural network with some weights W
- **Output**: A probability distribution over the next character

We can evaluate any setting of the parameters using our loss function (negative log likelihood). We'll use **gradient-based optimization** to tune the weights so that the neural net correctly predicts the probabilities for the next character.

### Why Can't We Just Input Integers?

You can't just plug an integer like 13 into a neural network. Neural nets are made up of neurons with weights that act **multiplicatively** on inputs: `wx + b`. It doesn't make sense for an input neuron to take on integer values.

### One-Hot Encoding

Instead, we use **one-hot encoding**: take an integer like 13 and create a vector that is all zeros except for the 13th position, which is 1.

```
13 → [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
```

Now this vector can feed into a neural net!

### The Forward Pass

We have:
1. **Input**: One-hot vector of shape (27,)
2. **Weights**: Matrix W of shape (27, 27) 
3. **Matrix multiply**: `logits = input @ W` → gives us (27,) numbers
4. **Exponentiate**: `counts = exp(logits)` → all positive numbers
5. **Normalize**: `probs = counts / sum(counts)` → probabilities that sum to 1

> **Softmax**: Steps 4-5 together are called **softmax**. It takes any real numbers (positive, negative, whatever) and outputs a valid probability distribution. This is the standard output layer for classification tasks.

In [ ]:
# Create training data: all bigrams as (input, target) pairs
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

print(f"Number of training examples: {num}")

### One-Hot Encoding

We represent each input character as a one-hot vector: a vector of zeros with a single 1 at the character's index.

In [ ]:
import torch.nn.functional as F

# One-hot encode all inputs
xenc = F.one_hot(xs, num_classes=27).float()
print(f"Shape of one-hot encoded inputs: {xenc.shape}")
print(f"First 5 inputs (character indices): {xs[:5].tolist()}")
print(f"First 5 targets (next characters): {ys[:5].tolist()}")

In [ ]:
# Visualize one-hot encoding for first 5 examples
plt.figure(figsize=(10, 3))
plt.imshow(xenc[:5].numpy())
plt.xlabel('Character index')
plt.ylabel('Example')
plt.title('One-hot encoding of first 5 input characters');

### The Forward Pass

Now let's implement the neural network forward pass.

In [ ]:
# Initialize random weights
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

print(f"Weight matrix shape: {W.shape}")

In [ ]:
# Forward pass
logits = xenc @ W              # (num, 27) @ (27, 27) -> (num, 27)
counts = logits.exp()          # exponentiate to get positive values
probs = counts / counts.sum(1, keepdims=True)  # normalize each row (softmax)

print(f"Logits shape: {logits.shape}")
print(f"Probs shape: {probs.shape}")
print(f"Each row sums to: {probs[0].sum().item():.4f}")

### Computing the Loss

The loss is the average negative log likelihood of the correct next character.

In [ ]:
# Get the probability the model assigns to the correct next character
# probs[i, ys[i]] is the probability of the correct character for example i
probs_correct = probs[torch.arange(num), ys]

# Compute negative log likelihood
loss = -probs_correct.log().mean()

print(f"Initial loss (random weights): {loss.item():.4f}")

> With random weights, the loss should be close to $\log(27) \approx 3.3$ (random guessing among 27 characters).

<a id='10-training'></a>
## 10. Training with Gradient Descent

Now we train the network using **gradient descent** - the same technique from micrograd:

1. **Forward pass**: Run inputs through network, compute loss
2. **Backward pass**: Compute gradients of loss w.r.t. weights (`loss.backward()`)
3. **Update**: Nudge weights in the direction that decreases loss
4. **Repeat**: Until loss is low enough

### Regularization = Smoothing

I want to point out an interesting equivalence. Remember **model smoothing** from the counting approach - where we added fake counts to make the distribution more uniform?

The neural network has an equivalent: **L2 regularization**.

Here's the insight:
- If all entries of W are **zero**, then logits are all zero
- `exp(0) = 1`, so all counts become 1
- Normalizing gives us **uniform probabilities** (1/27 each)

So making W closer to zero → more uniform distribution. We can **incentivize** this by adding a penalty term to the loss:

```python
regularization_loss = 0.01 * (W**2).mean()
total_loss = nll_loss + regularization_loss
```

This is called **L2 regularization** or **weight decay**:
- Squaring removes signs (positive and negative both become positive)
- If W is zero, regularization loss is zero
- If W has large values, you accumulate loss

The regularization strength (0.01) controls how strong the "pull toward zero" is:
- **Stronger regularization** → more uniform predictions (equivalent to adding more fake counts)
- **Weaker regularization** → more peaked predictions (trusting the data more)

This is kind of cool - the gradient-based framework has a natural equivalent to smoothing!

In [ ]:
# Re-initialize weights
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

# Training hyperparameters
learning_rate = 50
regularization = 0.01
num_iterations = 100

# Training loop
for k in range(num_iterations):
    # Forward pass
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    
    # Loss with L2 regularization
    loss = -probs[torch.arange(num), ys].log().mean() + regularization * (W**2).mean()
    
    # Backward pass
    W.grad = None
    loss.backward()
    
    # Update weights
    W.data += -learning_rate * W.grad
    
    # Print progress
    if k % 10 == 0 or k == num_iterations - 1:
        print(f"Iteration {k:3d}: loss = {loss.item():.4f}")

> **What just happened:** The loss decreased from ~3.8 (random weights, close to `log(27)≈3.3`) down to ~2.5 - approaching our counting-based model's performance! The neural network is learning the same bigram statistics, just via gradient descent instead of explicit counting.

<a id='11-neural-sampling'></a>
## 11. Sampling from the Trained Network

Let's generate names using our trained neural network.

### The Key Insight: One-Hot = Row Selection

I want you to notice something important. When we do:
```python
xenc @ W  # one-hot vector times weight matrix
```

If the one-hot vector has a 1 at position 5, then because of how matrix multiplication works, **we're just plucking out the 5th row of W**. The other zeros contribute nothing.

This is **exactly** what happened in our counting-based model! We took the first character and indexed into a row of our probability matrix. The neural network is doing the same thing - the weights W are essentially the **log counts**.

After training:
- `W` ≈ log of the count matrix
- `exp(W)` ≈ the count matrix itself

The difference is how we arrived there:
- **Counting**: Populated counts by explicitly counting bigrams
- **Neural net**: Initialized randomly, let the loss guide us to the same answer piece by piece

That's why both approaches give the same loss and the same samples!

In [ ]:
# Sample from the trained neural network
g = torch.Generator().manual_seed(2147483647)

print("Generated names (neural network):")
for i in range(10):
    out = []
    ix = 0  # start with '.'
    while True:
        # Neural network forward pass for single character
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdims=True)
        
        # Sample next character
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out[:-1]))

> **Kind of anticlimactic... or climactic?** We get the exact same results as the counting-based model! This isn't a bug - it's the point. These are **identical models**. Not only do they achieve the same loss, but `W` literally becomes the log counts. We just arrived at the same answer in a very different way.

<a id='12-conclusion'></a>
## 12. Conclusion & Next Steps

We've actually covered a lot of ground! Let me summarize what we did.

### What We Built

We introduced the **bigram character-level language model**. We saw:
- How to **train** the model (count bigrams and normalize)
- How to **sample** from the model (iteratively sample next character)
- How to **evaluate** the model (negative log likelihood loss)

Then we trained the model in **two completely different ways** that actually get the same result:

| Counting Approach | Neural Network Approach |
|-------------------|------------------------|
| Count bigrams explicitly | Learn via gradient descent |
| Normalize rows → probabilities | Softmax → probabilities |
| Smoothing = add fake counts | Regularization = push W toward 0 |
| Direct, interpretable | Flexible, scalable |

### Why The Neural Network Approach Matters

For bigrams, the counting approach is simpler. So why bother with neural networks?

Right now our neural network is **super simple**: single previous character → single linear layer → logits. But this is about to **complexify**.

In the follow-up videos, we're going to:
- Take **more and more** previous characters as context
- Feed them into increasingly sophisticated neural nets
- But the output is always the same: **logits** that get normalized the same way
- The loss function stays identical
- The gradient-based framework stays identical

It's just that the neural network will complexify all the way to **Transformers**.

### The Fundamental Limitation

The counting approach works for bigrams because there are only 27×27 = 729 possible pairs - we can store them all in a table.

But what about:
- **Trigrams?** 27³ = 19,683 combinations
- **4-grams?** 27⁴ = 531,441 combinations  
- **10-character context?** 27¹⁰ = way too many!

We can't keep everything in a table anymore. This is fundamentally an **unscalable approach**.

The neural network approach is significantly more scalable - it can **generalize** without storing every combination explicitly. That's where we'll be digging next.

### What's Next

- **Part 2**: MLP language models (more context)
- **Part 3**: Activations, gradients, BatchNorm
- **Part 4**: Building a WaveNet
- **Part 5**: Building a Transformer (GPT)

So that's going to be pretty awesome. I'm looking forward to it!

---

*Based on Andrej Karpathy's "Neural Networks: Zero to Hero" series. [Watch the original video](https://www.youtube.com/watch?v=PaCmpygFfXo)*